# Detección de Billetes — Ejecución Local en MacBook
## Proyecto de Innovación — Maestría en IA Aplicada

Este notebook corre **localmente en tu MacBook**, sin Colab ni internet.  
Usa el modelo `best.pt` entrenado en Colab y la cámara integrada o USB del Mac.

---

## 📋 Requisitos previos (hacer una sola vez en Terminal)

```bash
# 1. Crear entorno virtual
python3 -m venv ~/billetes_env
source ~/billetes_env/bin/activate

# 2. Instalar librerías
pip install ultralytics opencv-python jupyter matplotlib

# 3. Lanzar Jupyter
jupyter notebook
```

> **Coloca `best.pt`** en la misma carpeta que este notebook  
> o ajusta `RUTA_MODELO` en el Bloque 1.

---

| Bloque | Descripción |
|--------|-------------|
| **1** | Configuración: carga el modelo y verifica la cámara |
| **2** | Video en tiempo real con detección y total de dinero |
| **3** | Modo foto: captura y analiza una imagen individual |
| **4** | Solución de problemas comunes |

---
## Bloque 1 — Configuración

**¿Qué hace este bloque?**

- Verifica que las librerías estén instaladas
- Detecta el chip del Mac para usar **MPS** (GPU Apple Silicon) si está disponible — ~3x más rápido que CPU
- Carga el modelo `best.pt` entrenado
- Lista las cámaras disponibles
- Define los parámetros de detección ajustables


In [1]:
# ============================================================
# VERIFICACIÓN DEL ENTORNO Y CARGA DEL MODELO
# ============================================================
import cv2, torch, platform, os
from pathlib import Path
from ultralytics import YOLO

print('=' * 55)
print('INFORMACIÓN DEL SISTEMA')
print('=' * 55)
print(f'   Python  : {platform.python_version()}')
print(f'   Sistema : {platform.system()} {platform.machine()}')
print(f'   PyTorch : {torch.__version__}')
print(f'   OpenCV  : {cv2.__version__}')

# ── Selección del dispositivo ──
# MPS = Metal Performance Shaders, GPU del chip Apple (M1/M2/M3)
# Es ~3x más rápido que CPU para inferencia de redes neuronales
if torch.backends.mps.is_available():
    DEVICE = 'mps'
    print(f'\n✅ Apple MPS disponible — GPU Metal activa (chip Apple Silicon)')
elif torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f'\n✅ GPU NVIDIA: {torch.cuda.get_device_name(0)}')
else:
    DEVICE = 'cpu'
    print(f'\n⚠️  CPU — sin aceleración GPU (normal en Mac Intel)')

print(f'\n🔧 Dispositivo de inferencia: {DEVICE}')

# ============================================================
# CARGA DEL MODELO
# Ajusta RUTA_MODELO si best.pt está en otra ubicación:
#   './best.pt'                   -> misma carpeta que el notebook
#   '~/Downloads/best.pt'         -> carpeta Descargas
#   '~/Desktop/best.pt'           -> escritorio
# ============================================================
RUTA_MODELO = './best.pt'

if not Path(RUTA_MODELO).exists():
    print(f'\n❌ No se encontró el modelo en: {RUTA_MODELO}')
    print('   Copia best.pt a la misma carpeta que este notebook')
    print('   o cambia RUTA_MODELO con la ruta correcta')
    modelo = None
else:
    modelo = YOLO(RUTA_MODELO)
    n_params = sum(p.numel() for p in modelo.model.parameters())
    print(f'\n✅ Modelo cargado: {RUTA_MODELO}')
    print(f'   Clases    : {list(modelo.names.values())}')
    print(f'   Parámetros: {n_params:,}')


INFORMACIÓN DEL SISTEMA
   Python  : 3.11.11
   Sistema : Darwin arm64
   PyTorch : 2.11.0
   OpenCV  : 4.10.0

✅ Apple MPS disponible — GPU Metal activa (chip Apple Silicon)

🔧 Dispositivo de inferencia: mps

✅ Modelo cargado: ./best.pt
   Clases    : ['10000_pesos', '2000_pesos', '50000_pesos']
   Parámetros: 25,858,057


In [4]:
# ============================================================
# DETECCIÓN DE CÁMARAS DISPONIBLES
# OpenCV asigna índices: 0=cámara integrada, 1=primera USB, etc.
# ============================================================
print('🔍 Buscando cámaras disponibles...')
camaras_ok = []
for idx in range(5):
    cap = cv2.VideoCapture(idx)
    if cap.isOpened():
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        f = cap.get(cv2.CAP_PROP_FPS)
        nombre = 'FaceTime (integrada)' if idx == 0 else f'Cámara USB #{idx}'
        print(f'   [{idx}] ✅ {nombre} — {w}×{h} @ {f:.0f}fps')
        camaras_ok.append(idx)
    cap.release()

if not camaras_ok:
    print('   ❌ Sin cámaras — verifica permisos:')
    print('   Apple → Config. del Sistema → Privacidad → Cámara → Terminal')

# ── Parámetros de detección (modifica según resultados) ──
INDICE_CAMARA = 0    # 0=integrada, 1=USB externa

CONFIANZA = 0.50     # Umbral de confianza mínima para reportar detección
                     # Sube si ves detecciones falsas (ej: 0.55)
                     # Baja si el modelo no detecta los billetes (ej: 0.30)

IOU_THRESH = 0.30    # Umbral para eliminar cajas duplicadas del mismo objeto
                     # 0.45 es el valor óptimo para billetes

print(f'\n⚙️  Parámetros:')
print(f'   INDICE_CAMARA = {INDICE_CAMARA}')
print(f'   CONFIANZA     = {CONFIANZA}')
print(f'   IOU_THRESH    = {IOU_THRESH}')
print(f'\n✅ Listo — ejecuta el Bloque 2 para iniciar el video')


🔍 Buscando cámaras disponibles...
   [0] ✅ FaceTime (integrada) — 1920×1080 @ 24fps
   [1] ✅ Cámara USB #1 — 1920×1080 @ 30fps
   [2] ✅ Cámara USB #2 — 1920×1080 @ 30fps
[05/08 20:40:20.303512][info][7127300][Context.cpp:69] Context created with config: default config!
[05/08 20:40:20.303566][info][7127300][Context.cpp:74] Context work_dir=/Users/joseluis/Documents/Maestria_IA_Aplicada/Semestre_II/ProyectoSemestre_2/Sesion_3/Tarea_2
[05/08 20:40:20.303570][info][7127300][Context.cpp:77] 	- SDK version: 1.9.4
[05/08 20:40:20.303571][info][7127300][Context.cpp:78] 	- SDK stage version: main
[05/08 20:40:20.303575][info][7127300][Context.cpp:82] get config EnumerateNetDevice:false
[05/08 20:40:20.303577][info][7127300][MacPal.cpp:36] createObPal: create MacPal!
[05/08 20:40:20.309584][info][7127300][MacPal.cpp:104] Create PollingDeviceWatcher!
[05/08 20:40:20.309596][info][7127300][DeviceManager.cpp:15] Current found device(s): (0)
[05/08 20:40:20.309629][info][7127300][Pipeline.cpp:15] T

OpenCV: out device of bound (0-2): 3
OpenCV: camera failed to properly initialize!
[ WARN:0@78.581] global cap.cpp:323 open VIDEOIO(OBSENSOR): raised unknown C++ exception!


OpenCV: out device of bound (0-2): 4
OpenCV: camera failed to properly initialize!
[ WARN:0@78.599] global cap.cpp:323 open VIDEOIO(OBSENSOR): raised unknown C++ exception!




---
## Bloque 2 — Video en Tiempo Real

**¿Qué hace este bloque?**

Abre una ventana de video (fuera del notebook) con detección en cada frame.  
Muestra el bounding box, denominación, confianza y el total acumulado.

**Teclas durante la ejecución:**

| Tecla | Acción |
|-------|--------|
| `q` | Cerrar el video |
| `s` | Guardar captura del frame actual |
| `+` | Subir umbral de confianza |
| `-` | Bajar umbral de confianza |
| `p` | Pausar / reanudar |

> ⚠️ La ventana abre **fuera del notebook** — si no aparece, presiona `Cmd+Tab`


In [5]:
# ============================================================
# DETECCIÓN EN VIDEO EN TIEMPO REAL
#
# Por cada frame el bucle:
#   1. Captura el frame de la cámara
#   2. Envía el frame a YOLOv8 para inferencia
#   3. Dibuja bounding boxes y etiquetas
#   4. Calcula el total de dinero en escena
#   5. Muestra FPS y controles en pantalla
#   6. Procesa teclas del teclado
# ============================================================
import cv2, time
from pathlib import Path

if modelo is None:
    print('❌ Primero ejecuta el Bloque 1 para cargar el modelo')
else:

    # Colores por denominación (formato BGR de OpenCV)
    COLORES = {
        '2000_pesos':  (255, 140,   0),   # Naranja
        '10000_pesos': (  0, 200,   0),   # Verde
        '50000_pesos': (  0, 165, 255),   # Azul claro
    }

    # Función auxiliar: extrae valor de '10000_pesos' -> 10000
    def valor_clase(nombre):
        try: return int(nombre.split('_pesos')[0])
        except: return 0

    camara = cv2.VideoCapture(INDICE_CAMARA)

    if not camara.isOpened():
        print(f'❌ No se pudo abrir la cámara {INDICE_CAMARA}')
        print('   Verifica: Apple -> Config. del Sistema -> Privacidad -> Cámara')
    else:
        camara.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
        camara.set(cv2.CAP_PROP_FRAME_HEIGHT,  720)
        camara.set(cv2.CAP_PROP_FPS, 30)
        w = int(camara.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(camara.get(cv2.CAP_PROP_FRAME_HEIGHT))
        print(f'✅ Cámara abierta: {w}x{h}')
        print('   Teclas: q=salir  s=guardar  +/-=confianza  p=pausa')
        print('   La primera detección tarda 2-3s (MPS compilando kernels)')

        conf_actual = CONFIANZA
        pausado     = False
        n_capturas  = 0
        t_anterior  = time.time()
        fps_val     = 0.0
        n_frames    = 0

        while True:
            if not pausado:
                ok, frame = camara.read()
                if not ok:
                    print('⚠️  Error leyendo frame'); break

                n_frames += 1

                # ── Inferencia YOLOv8 ──
                resultados = modelo.predict(
                    frame, conf=conf_actual, iou=IOU_THRESH,
                    device=DEVICE, verbose=False, stream=False)

                total = 0
                n_det = 0
                for res in resultados:
                    for box in res.boxes:
                        cid   = int(box.cls[0])
                        nom   = modelo.names[cid]
                        conf_ = float(box.conf[0])
                        val   = valor_clase(nom)
                        total += val
                        n_det += 1

                        x1,y1,x2,y2 = map(int, box.xyxy[0])
                        col = COLORES.get(nom, (0,255,255))
                        cv2.rectangle(frame,(x1,y1),(x2,y2),col,3)

                        lbl = f'${val:,}  {conf_:.0%}'
                        (tw,th),_ = cv2.getTextSize(lbl,cv2.FONT_HERSHEY_SIMPLEX,0.75,2)
                        yt = max(y1-10, th+10)
                        cv2.rectangle(frame,(x1,yt-th-8),(x1+tw+8,yt+4),col,-1)
                        cv2.putText(frame,lbl,(x1+4,yt),
                            cv2.FONT_HERSHEY_SIMPLEX,0.75,(255,255,255),2)

                # ── Panel superior ──
                if n_frames % 10 == 0:
                    ahora = time.time()
                    fps_val = 10.0 / (ahora - t_anterior + 1e-9)
                    t_anterior = ahora

                overlay = frame.copy()
                cv2.rectangle(overlay,(0,0),(w,72),(0,0,0),-1)
                cv2.addWeighted(overlay,0.6,frame,0.4,0,frame)

                l1 = f'TOTAL: ${total:,} pesos  ({n_det} billete{"s" if n_det!=1 else ""})'
                l2 = f'FPS:{fps_val:.1f}  Conf:{conf_actual:.2f}  {"PAUSADO" if pausado else "EN VIVO"}'
                cv2.putText(frame,l1,(10,36),cv2.FONT_HERSHEY_SIMPLEX,1.1,(0,255,255),2)
                cv2.putText(frame,l2,(10,63),cv2.FONT_HERSHEY_SIMPLEX,0.6,(180,180,180),1)
                cv2.putText(frame,'q=salir  s=guardar  +/-=confianza  p=pausa',
                    (10,h-10),cv2.FONT_HERSHEY_SIMPLEX,0.5,(130,130,130),1)

            cv2.imshow('Detector de Billetes Colombianos', frame)
            tecla = cv2.waitKey(1) & 0xFF

            if   tecla == ord('q'):
                print('\n👋 Cerrando...'); break
            elif tecla == ord('s') and not pausado:
                fname = f'captura_{n_capturas:03d}_total{total}.jpg'
                cv2.imwrite(fname, frame)
                n_capturas += 1
                print(f'📸 Guardado: {fname}')
            elif tecla == ord('+'):
                conf_actual = min(0.95, round(conf_actual+0.05,2))
                print(f'⚙️  Confianza -> {conf_actual:.2f}')
            elif tecla == ord('-'):
                conf_actual = max(0.05, round(conf_actual-0.05,2))
                print(f'⚙️  Confianza -> {conf_actual:.2f}')
            elif tecla == ord('p'):
                pausado = not pausado
                print('⏸️  Pausado' if pausado else '▶️  Reanudado')

        camara.release()
        cv2.destroyAllWindows()
        print(f'✅ Video cerrado  |  capturas guardadas: {n_capturas}')


✅ Cámara abierta: 1280x720
   Teclas: q=salir  s=guardar  +/-=confianza  p=pausa
   La primera detección tarda 2-3s (MPS compilando kernels)

👋 Cerrando...
✅ Video cerrado  |  capturas guardadas: 0


---
## Bloque 3 — Modo Foto

Captura **una sola foto** con la cámara y la analiza en detalle.  
El resultado se muestra dentro del notebook con matplotlib.  
Útil para documentar el resultado o incluirlo en el informe.

También puedes analizar una **imagen existente** (de archivo)
cambiando `USAR_IMAGEN_EXISTENTE = True`.


In [ ]:
# ============================================================
# CAPTURA Y ANÁLISIS DE FOTO INDIVIDUAL
# ============================================================
import cv2, time
import matplotlib.pyplot as plt
from pathlib import Path

SEGUNDOS_ESPERA       = 3       # Tiempo antes de capturar
NOMBRE_SALIDA         = 'analisis_billetes.jpg'
USAR_IMAGEN_EXISTENTE = False   # True = analiza archivo existente
RUTA_IMAGEN_EXISTENTE = './mi_imagen.jpg'

def analizar_imagen(img, mod, conf=0.45, iou=0.45):
    COLORES = {
        '2000_pesos': (255,140,0),
        '10000_pesos':(0,200,0),
        '50000_pesos':(0,165,255)
    }
    res_list = mod.predict(img, conf=conf, iou=iou, device=DEVICE, verbose=False)
    total, detalle = 0, []
    for res in res_list:
        for box in res.boxes:
            cid  = int(box.cls[0])
            nom  = mod.names[cid]
            conf_= float(box.conf[0])
            try:    val = int(nom.split('_pesos')[0])
            except: val = 0
            total += val
            detalle.append({'clase':nom,'valor':val,'confianza':conf_})
            x1,y1,x2,y2 = map(int,box.xyxy[0])
            col = COLORES.get(nom,(0,255,255))
            cv2.rectangle(img,(x1,y1),(x2,y2),col,3)
            lbl = f'${val:,}  {conf_:.0%}'
            (tw,th),_ = cv2.getTextSize(lbl,cv2.FONT_HERSHEY_SIMPLEX,0.9,2)
            yt = max(y1-10,th+10)
            cv2.rectangle(img,(x1,yt-th-10),(x1+tw+10,yt+5),col,-1)
            cv2.putText(img,lbl,(x1+5,yt),cv2.FONT_HERSHEY_SIMPLEX,0.9,(255,255,255),2)
    # Panel con total
    txt = f'TOTAL: ${total:,} pesos'
    (pw,ph),_ = cv2.getTextSize(txt,cv2.FONT_HERSHEY_SIMPLEX,1.3,3)
    cv2.rectangle(img,(0,0),(pw+20,ph+25),(0,0,0),-1)
    cv2.putText(img,txt,(10,ph+10),cv2.FONT_HERSHEY_SIMPLEX,1.3,(0,255,255),3)
    return img, total, detalle

# ── Obtener imagen ──
if modelo is None:
    print('❌ Primero ejecuta el Bloque 1')
elif USAR_IMAGEN_EXISTENTE:
    if not Path(RUTA_IMAGEN_EXISTENTE).exists():
        print(f'❌ No encontrado: {RUTA_IMAGEN_EXISTENTE}')
        imagen = None
    else:
        imagen = cv2.imread(RUTA_IMAGEN_EXISTENTE)
        print(f'✅ Imagen cargada desde archivo')
else:
    cam = cv2.VideoCapture(INDICE_CAMARA)
    if not cam.isOpened():
        print('❌ No se pudo abrir la cámara'); imagen = None
    else:
        cam.set(cv2.CAP_PROP_FRAME_WIDTH,1280)
        cam.set(cv2.CAP_PROP_FRAME_HEIGHT,720)
        print(f'📷 Capturando en {SEGUNDOS_ESPERA} segundos...')
        print('   Posiciona los billetes frente a la cámara ahora.')
        t0 = time.time()
        while time.time()-t0 < SEGUNDOS_ESPERA:
            ok,frm = cam.read()
            if ok:
                r = SEGUNDOS_ESPERA - int(time.time()-t0)
                cv2.putText(frm,f'Capturando en {r}s...',(30,60),
                    cv2.FONT_HERSHEY_SIMPLEX,1.5,(0,255,255),3)
                cv2.imshow('Preview',frm); cv2.waitKey(1)
        ok,imagen = cam.read()
        cam.release(); cv2.destroyAllWindows()
        if not ok: print('❌ Error capturando'); imagen = None
        else: print('✅ Foto capturada')

# ── Análisis y visualización ──
if 'imagen' in dir() and imagen is not None:
    print('🔍 Analizando...')
    img_anot, total, detalle = analizar_imagen(
        imagen.copy(), modelo, CONFIANZA, IOU_THRESH)
    cv2.imwrite(NOMBRE_SALIDA, img_anot)

    # Mostramos dentro del notebook con matplotlib
    # (convertimos BGR→RGB porque matplotlib usa RGB)
    img_rgb = cv2.cvtColor(img_anot, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(13,7))
    ax.imshow(img_rgb)
    ax.axis('off')
    ax.set_title(f'Resultado: TOTAL ${total:,} pesos', fontsize=15, pad=10)
    plt.tight_layout()
    plt.show()

    print('\n' + '='*48)
    print('RESULTADO')
    print('='*48)
    if detalle:
        for d in detalle:
            print(f'   ${d["valor"]:>10,}  confianza: {d["confianza"]:.1%}')
        print(f'   {chr(8212)*35}')
        print(f'   TOTAL: ${total:>10,} pesos')
    else:
        print('   ⚠️  Sin detecciones')
        print(f'   Prueba bajando CONFIANZA (actual={CONFIANZA})')
        print('   o mejora la iluminación')
    print(f'\n📁 Imagen guardada: {NOMBRE_SALIDA}')


---
## Solución de Problemas Comunes

### La ventana de video no abre
macOS requiere permiso explícito para que Terminal acceda a la cámara:  
`Apple → Configuración del Sistema → Privacidad y Seguridad → Cámara → Terminal ✅`

### `cv2.error: (-215) !_src.empty()`
La cámara está ocupada (Zoom, FaceTime, Meet) o el índice es incorrecto.  
Cierra otras apps y prueba con `INDICE_CAMARA = 1`.

### El modelo no detecta los billetes
1. Baja `CONFIANZA` a `0.30` en el Bloque 1
2. Mejora la iluminación — evita reflejos directos
3. Mantén el billete a 20-40 cm de la cámara
4. Verifica que `RUTA_MODELO` apunte al `best.pt` correcto

### Muy lento en Mac Intel
El chip Intel no tiene MPS. Opciones para acelerar:
```python
# Reducir tamaño de imagen en la inferencia (en el bucle de video):
resultados = modelo.predict(frame, imgsz=320, ...)  # en lugar de 640
```

### `ModuleNotFoundError: No module named 'ultralytics'`
El entorno virtual no está activado. En Terminal:
```bash
source ~/billetes_env/bin/activate
jupyter notebook
```

### La ventana de video se congela
`cv2.waitKey(1)` es obligatorio en el bucle para que OpenCV actualice  
la ventana. Si eliminaste esa línea, restáurala.
